In [1]:
def predict_dropout1():
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    train['test_1_num']    = train['test 1'].str.upper().map(GRADE_MAP)
    train['forum_Q_s1']    = train['forum Q'] / 4
    train['forum_A_s1']    = train['forum A'] / 4
    train['oh_s1']         = train['office hour visits'] / 4
    train['session_total'] = train['session 1'].fillna(0) + train['session 2'].fillna(0)
    train['grade_mean']    = train['test_1_num']
    train['y']             = train['dropout'].map({'Y': 1, 'N': 0})

    feats = ['external_flag', 'year_num', 'session 1', 'session 2',
             'test_1_num', 'forum_Q_s1', 'forum_A_s1', 'oh_s1',
             'session_total', 'grade_mean']

    imp    = SimpleImputer(strategy='median')
    X_tr   = imp.fit_transform(train[feats].values)
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    y_tr   = train['y'].values

    model = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5, random_state=42)
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout1.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    test['test_1_num']    = test['test 1'].str.upper().map(GRADE_MAP)
    test['forum_Q_s1']    = test['forum Q'] / 4
    test['forum_A_s1']    = test['forum A'] / 4
    test['oh_s1']         = test['office hour visits'] / 4
    test['session_total'] = test['session 1'].fillna(0) + test['session 2'].fillna(0)
    test['grade_mean']    = test['test_1_num']

    X_te   = scaler.transform(imp.transform(test[feats].values))
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [2]:
def predict_dropout2():
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer

    GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2']:
        train[col.replace(' ', '_') + '_num'] = train[col].str.upper().map(GRADE_MAP)
    train['forum_Q_s2']    = train['forum Q'] / 2
    train['forum_A_s2']    = train['forum A'] / 2
    train['oh_s2']         = train['office hour visits'] / 2
    train['session_total'] = (train['session 1'].fillna(0) + train['session 2'].fillna(0) +
                              train['session 3'].fillna(0) + train['session 4'].fillna(0))
    train['grade_mean']    = train[['test_1_num', 'test_2_num']].mean(axis=1)
    train['grade_delta']   = train['test_2_num'] - train['test_1_num']
    train['y']             = train['dropout'].map({'Y': 1, 'N': 0})

    train = train[train['session 3'].notna() | train['test 2'].notna()]

    feats = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4',
             'test_1_num', 'test_2_num', 'forum_Q_s2', 'forum_A_s2', 'oh_s2',
             'session_total', 'grade_mean', 'grade_delta']

    imp  = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(train[feats].values)
    y_tr = train['y'].values

    model = RandomForestClassifier(n_estimators=300, max_depth=6,
                                   class_weight='balanced_subsample',
                                   random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout2.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2']:
        test[col.replace(' ', '_') + '_num'] = test[col].str.upper().map(GRADE_MAP)
    test['forum_Q_s2']    = test['forum Q'] / 2
    test['forum_A_s2']    = test['forum A'] / 2
    test['oh_s2']         = test['office hour visits'] / 2
    test['session_total'] = (test['session 1'].fillna(0) + test['session 2'].fillna(0) +
                             test['session 3'].fillna(0) + test['session 4'].fillna(0))
    test['grade_mean']    = test[['test_1_num', 'test_2_num']].mean(axis=1)
    test['grade_delta']   = test['test_2_num'] - test['test_1_num']

    X_te   = imp.transform(test[feats].values)
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [3]:
def predict_dropout3():
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer

    GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2', 'test 3']:
        train[col.replace(' ', '_') + '_num'] = train[col].str.upper().map(GRADE_MAP)
    train['forum_Q_s3']    = train['forum Q'] * 0.75
    train['forum_A_s3']    = train['forum A'] * 0.75
    train['oh_s3']         = train['office hour visits'] * 0.75
    train['session_total'] = (train['session 1'].fillna(0) + train['session 2'].fillna(0) +
                              train['session 3'].fillna(0) + train['session 4'].fillna(0) +
                              train['session 5'].fillna(0))
    train['grade_mean']    = train[['test_1_num', 'test_2_num', 'test_3_num']].mean(axis=1)
    train['grade_delta']   = train['test_3_num'] - train['test_1_num']
    train['y']             = train['dropout'].map({'Y': 1, 'N': 0})

    train = train[train['session 5'].notna() | train['test 3'].notna()]

    feats = ['external_flag', 'year_num',
             'session 1', 'session 2', 'session 3', 'session 4', 'session 5',
             'test_1_num', 'test_2_num', 'test_3_num',
             'forum_Q_s3', 'forum_A_s3', 'oh_s3',
             'session_total', 'grade_mean', 'grade_delta']

    imp  = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(train[feats].values)
    y_tr = train['y'].values

    model = HistGradientBoostingClassifier(max_iter=200, max_depth=5,
                                           class_weight='balanced', random_state=42)
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout3.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2', 'test 3']:
        test[col.replace(' ', '_') + '_num'] = test[col].str.upper().map(GRADE_MAP)
    test['forum_Q_s3']    = test['forum Q'] * 0.75
    test['forum_A_s3']    = test['forum A'] * 0.75
    test['oh_s3']         = test['office hour visits'] * 0.75
    test['session_total'] = (test['session 1'].fillna(0) + test['session 2'].fillna(0) +
                             test['session 3'].fillna(0) + test['session 4'].fillna(0) +
                             test['session 5'].fillna(0))
    test['grade_mean']    = test[['test_1_num', 'test_2_num', 'test_3_num']].mean(axis=1)
    test['grade_delta']   = test['test_3_num'] - test['test_1_num']

    X_te   = imp.transform(test[feats].values)
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [4]:
def predict_dropout4():
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import (RandomForestClassifier,
                                  HistGradientBoostingClassifier,
                                  StackingClassifier)
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    GRADE_MAP  = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP   = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}
    GRADE_COLS = ['test 1', 'test 2', 'test 3', 'ind cw', 'group cw', 'final grade']

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    for col in GRADE_COLS:
        train[col.replace(' ', '_') + '_num'] = train[col].str.upper().map(GRADE_MAP)
    train['forum_Q_s4']    = train['forum Q']
    train['forum_A_s4']    = train['forum A']
    train['oh_s4']         = train['office hour visits']
    train['session_total'] = (train['session 1'].fillna(0) + train['session 2'].fillna(0) +
                              train['session 3'].fillna(0) + train['session 4'].fillna(0) +
                              train['session 5'].fillna(0) + train['session 6'].fillna(0))
    num_grade_cols = ['test_1_num', 'test_2_num', 'test_3_num',
                      'ind_cw_num', 'group_cw_num', 'final_grade_num']
    train['grade_mean']  = train[num_grade_cols].mean(axis=1)
    train['grade_delta'] = train['final_grade_num'] - train['test_1_num']
    train['y']           = train['dropout'].map({'Y': 1, 'N': 0})

    train = train[train['session 6'].notna() | train['ind cw'].notna()]

    feats = ['external_flag', 'year_num',
             'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'session 6',
             'test_1_num', 'test_2_num', 'test_3_num',
             'ind_cw_num', 'group_cw_num', 'final_grade_num',
             'forum_Q_s4', 'forum_A_s4', 'oh_s4',
             'session_total', 'grade_mean', 'grade_delta']

    imp    = SimpleImputer(strategy='median')
    X_tr   = imp.fit_transform(train[feats].values)
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    y_tr   = train['y'].values

    model = StackingClassifier(
        estimators=[
            ('lr',  LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced')),
            ('rf',  RandomForestClassifier(n_estimators=200, random_state=42,
                                           class_weight='balanced_subsample', n_jobs=-1)),
            ('hgb', HistGradientBoostingClassifier(max_iter=150, random_state=42)),
        ],
        final_estimator=LogisticRegression(C=1.0),
        passthrough=False, cv=3
    )
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout4.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    for col in GRADE_COLS:
        test[col.replace(' ', '_') + '_num'] = test[col].str.upper().map(GRADE_MAP)
    test['forum_Q_s4']    = test['forum Q']
    test['forum_A_s4']    = test['forum A']
    test['oh_s4']         = test['office hour visits']
    test['session_total'] = (test['session 1'].fillna(0) + test['session 2'].fillna(0) +
                             test['session 3'].fillna(0) + test['session 4'].fillna(0) +
                             test['session 5'].fillna(0) + test['session 6'].fillna(0))
    test['grade_mean']  = test[num_grade_cols].mean(axis=1)
    test['grade_delta'] = test['final_grade_num'] - test['test_1_num']

    X_te   = scaler.transform(imp.transform(test[feats].values))
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [5]:
print("Stage 1:", predict_dropout1())
print("Stage 2:", predict_dropout2())
print("Stage 3:", predict_dropout3())
print("Stage 4:", predict_dropout4())

Stage 1: ['Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'N', 'N', 'Y']
Stage 2: ['Y', 'Y', 'N', 'N', 'N', 'N', 'Y', 'Y', 'Y', 'N']
Stage 3: ['Y', 'Y', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'N', 'N']
Stage 4: ['N', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'Y', 'N']


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Load actual labels from training data
train = pd.read_csv('ND26_dropout.csv')
y_actual = train['dropout'].map({'Y': 1, 'N': 0}).values

# Get predictions from each stage
pred1 = [1 if p == 'Y' else 0 for p in predict_dropout1()]
pred2 = [1 if p == 'Y' else 0 for p in predict_dropout2()]
pred3 = [1 if p == 'Y' else 0 for p in predict_dropout3()]
pred4 = [1 if p == 'Y' else 0 for p in predict_dropout4()]

# Generate reports for each stage
for stage, predictions in enumerate([pred1, pred2, pred3, pred4], 1):
    print(f"\n{'='*50}")
    print(f"Stage {stage} Results")
    print(f"{'='*50}")
    # load actual labels for the corresponding stage test file
    test_actual = pd.read_csv(f'./data/entry_dropout{stage}.csv')
    y_true = test_actual['dropout'].map({'Y': 1, 'N': 0}).values
    print(f"Accuracy: {accuracy_score(y_true, predictions):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_actual, predictions, target_names=['No Dropout', 'Dropout']))


Stage 1 Results


ValueError: Found input variables with inconsistent numbers of samples: [25749, 10]